In [1]:
# ============================================================
# exp_20260428_061_ashish_xgb_orig_te_bias
# Purpose:
#   Ashish Singh Rawat shared XGB LB0.98109 notebook reproduction
#   - single submission
#   - OOF / pred npy save
#   - bias tuning
#   - corr vs existing OOF
# ============================================================

In [2]:
from __future__ import annotations

import os
import gc
import json
import random
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import torch

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260428_061_ashish_xgb_orig_te_bias"
    EXP_NAME = "ashish_xgb_orig_te_bias"

    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    SAMPLE_PATH = "/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv"
    ORIG_PATH = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

    TARGET = "Irrigation_Need"
    ID_COL = "id"

    SEED = 42
    N_SPLITS = 5
    ORIG_ROW_WEIGHT = 0.5

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    XGB_PARAMS = {
        "objective": "multi:softprob",
        "num_class": 3,
        "n_estimators": 2600,
        "learning_rate": 0.05,
        "max_depth": 3,
        "subsample": 0.9,
        "colsample_bytree": 0.8,
        "max_bin": 1100,
        "eval_metric": "mlogloss",
        "n_jobs": -1,
        "random_state": SEED,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "enable_categorical": True,
    }

    REF_NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"
    REF_OOF = {
        "018": REF_NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy",
        "046b": REF_NPY_PATH + "oof_xgb_digit_orderedte_min_optuna_biased_refined.npy",
        "025": REF_NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy",
        "053": REF_NPY_PATH + "oof_lgb_digit_te_threshold_min_from_family_proba_biased.npy",
        "056": REF_NPY_PATH + "oof_cat_pairwise_te_threshold_min_from_family_gpu_default_fix_proba_biased.npy",
        "057": REF_NPY_PATH + "oof_xgb046b_high_interaction_min_biased.npy",
        "060": REF_NPY_PATH + "oof_cat_depth1_formula_min_biased.npy",
    }

In [4]:
# ============================================================
# Utils
# ============================================================

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def public_preds(proba: np.ndarray, bias: np.ndarray) -> np.ndarray:
    return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias.reshape(1, -1), axis=1)


def tune_bias_coordinate_descent(
    proba: np.ndarray,
    y_true: np.ndarray,
    steps=(1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002),
):
    best_bias = np.zeros(proba.shape[1], dtype=np.float64)
    best_score = balanced_accuracy_score(y_true, public_preds(proba, best_bias))
    history = [best_score]

    for step in steps:
        improved = True
        while improved:
            improved = False
            for ci in range(proba.shape[1]):
                for direction in (-1.0, 1.0):
                    cand = best_bias.copy()
                    cand[ci] += direction * step
                    score = balanced_accuracy_score(y_true, public_preds(proba, cand))
                    if score > best_score + 1e-9:
                        best_bias = cand
                        best_score = score
                        improved = True
                        history.append(best_score)
                        print(f"bias step={step:.4f} score={best_score:.9f} bias={best_bias}")

    return best_bias, float(best_score), history


def apply_bias_to_proba(proba: np.ndarray, bias: np.ndarray) -> np.ndarray:
    logp = np.log(np.clip(proba, 1e-15, 1.0)) + bias.reshape(1, -1)
    logp = logp - logp.max(axis=1, keepdims=True)
    e = np.exp(logp)
    return (e / e.sum(axis=1, keepdims=True)).astype(np.float32)


def mean_raw_corr(a, b):
    vals = []
    for c in range(a.shape[1]):
        vals.append(np.corrcoef(a[:, c], b[:, c])[0, 1])
    return float(np.mean(vals))


def mean_rank_corr(a, b):
    vals = []
    for c in range(a.shape[1]):
        ra = pd.Series(a[:, c]).rank(method="average").to_numpy()
        rb = pd.Series(b[:, c]).rank(method="average").to_numpy()
        vals.append(np.corrcoef(ra, rb)[0, 1])
    return float(np.mean(vals))


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
sample = pd.read_csv(CFG.SAMPLE_PATH)
orig = pd.read_csv(CFG.ORIG_PATH)

print("train:", train.shape)
print("test :", test.shape)
print("orig :", orig.shape)

target = CFG.TARGET

cat_cols = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Mulching_Used",
    "Region",
]

num_cols = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

top_cat_cols = ["Crop_Growth_Stage", "Mulching_Used", "Crop_Type"]
top_num_cols = ["Soil_Moisture", "Temperature_C", "Wind_Speed_kmh", "Rainfall_mm"]
top_cols = [
    "Soil_Moisture",
    "Crop_Growth_Stage",
    "Temperature_C",
    "Mulching_Used",
    "Wind_Speed_kmh",
    "Rainfall_mm",
]

for df in [train, orig]:
    df[target] = df[target].map(CFG.TARGET_MAP).astype(int)

train_fe = train.copy()
test_fe = test.copy()
orig_fe = orig.copy()

train: (630000, 21)
test : (270000, 20)
orig : (10000, 20)


In [6]:
# ============================================================
# Feature Engineering
# ============================================================

def add_threshold_distances(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["soil_lt_25"] = (df["Soil_Moisture"] < 25).astype(int)
    df["wind_gt_10"] = (df["Wind_Speed_kmh"] > 10).astype(int)
    df["temp_gt_30"] = (df["Temperature_C"] > 30).astype(int)
    df["rain_lt_300"] = (df["Rainfall_mm"] < 300).astype(int)

    df["moist_rain"] = df["Soil_Moisture"] / (df["Rainfall_mm"] + 1)
    df["moist_temp"] = df["Soil_Moisture"] / (df["Temperature_C"] + 1)
    df["moist_wind"] = df["Soil_Moisture"] / (df["Wind_Speed_kmh"] + 1)
    df["ET_proxy"] = (
        df["Temperature_C"] * df["Wind_Speed_kmh"] * df["Sunlight_Hours"]
    ) / (df["Humidity"] + 1)
    df["heat_stress"] = df["Temperature_C"] * df["Sunlight_Hours"]
    df["dfrying_force"] = df["Wind_Speed_kmh"] * df["Temperature_C"] / (df["Humidity"] + 1)
    df["water_supply"] = df["Rainfall_mm"] + df["Previous_Irrigation_mm"]
    df["water_dfeficit"] = df["Soil_Moisture"] - df["water_supply"] * 0.1
    df["soil_quality"] = df["Organic_Carbon"] / (df["Electrical_Conductivity"] + 0.1)
    df["moist_x_temp"] = df["Soil_Moisture"] * df["Temperature_C"]
    df["windf_x_temp"] = df["Wind_Speed_kmh"] * df["Temperature_C"]

    return df


def add_formula_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["high_score"] = (
        (df["Soil_Moisture"] < 25) * 2
        + (df["Rainfall_mm"] < 300) * 2
        + (df["Temperature_C"] > 30) * 1
        + (df["Wind_Speed_kmh"] > 10) * 1
    )

    df["low_score"] = (
        (df["Crop_Growth_Stage"].isin(["Harvest", "Sowing"])) * 2
        + (df["Mulching_Used"] == "Yes") * 1
    )

    df["formula_score"] = df["high_score"] - df["low_score"]

    df["formula_pred"] = 0
    df.loc[(df["formula_score"] > 0) & (df["formula_score"] <= 3), "formula_pred"] = 1
    df.loc[df["formula_score"] > 3, "formula_pred"] = 2

    return df


for name, df in [("train", train_fe), ("test", test_fe), ("orig", orig_fe)]:
    df2 = add_threshold_distances(df)
    df2 = add_formula_features(df2)
    if name == "train":
        train_fe = df2
    elif name == "test":
        test_fe = df2
    else:
        orig_fe = df2

In [7]:
# ------------------------------------------------------------
# N-gram features
# ------------------------------------------------------------

BIGRAM_COLS = []
TRIGRAM_COLS = []

for c1, c2 in combinations(top_cat_cols, 2):
    col_name = f"BG_{c1}_{c2}"
    for df in [train_fe, test_fe, orig_fe]:
        df[col_name] = df[c1].astype(str) + "_" + df[c2].astype(str)
    BIGRAM_COLS.append(col_name)

for c1, c2, c3 in combinations(top_cat_cols, 3):
    col_name = f"TG_{c1}_{c2}_{c3}"
    for df in [train_fe, test_fe, orig_fe]:
        df[col_name] = df[c1].astype(str) + "_" + df[c2].astype(str) + "_" + df[c3].astype(str)
    TRIGRAM_COLS.append(col_name)

NGRAM_COLS = BIGRAM_COLS + TRIGRAM_COLS

for col in NGRAM_COLS:
    combined = pd.concat([train_fe[col], test_fe[col], orig_fe[col]], axis=0).astype(str)
    labels, uniques = pd.factorize(combined)
    n_train = len(train_fe)
    n_test = len(test_fe)

    train_fe[col] = labels[:n_train]
    test_fe[col] = labels[n_train:n_train + n_test]
    orig_fe[col] = labels[n_train + n_test:]

print("NGRAM_COLS:", NGRAM_COLS)

NGRAM_COLS: ['BG_Crop_Growth_Stage_Mulching_Used', 'BG_Crop_Growth_Stage_Crop_Type', 'BG_Mulching_Used_Crop_Type', 'TG_Crop_Growth_Stage_Mulching_Used_Crop_Type']


In [8]:
# ------------------------------------------------------------
# num & cat interactions
# ------------------------------------------------------------

BIN_CAT_INT_COLS = []

for num_col in top_num_cols:
    bin_col = f"{num_col}_bin"
    train_fe[bin_col], bins = pd.qcut(
        train_fe[num_col],
        q=5,
        labels=False,
        retbins=True,
        duplicates="drop",
    )

    for df in [test_fe, orig_fe]:
        df[bin_col] = (
            pd.cut(df[num_col], bins=bins, labels=False, include_lowest=True)
            .fillna(0)
            .astype(int)
        )

    for cat_col in top_cat_cols:
        int_name = f"{num_col}_bin_{cat_col}_int"
        for df in [train_fe, test_fe, orig_fe]:
            df[int_name] = df[bin_col].astype(str) + "_" + df[cat_col].astype(str)
        BIN_CAT_INT_COLS.append(int_name)

for df in [train_fe, test_fe, orig_fe]:
    df.drop(columns=[f"{num_col}_bin" for num_col in top_num_cols], inplace=True)

for col in BIN_CAT_INT_COLS:
    codes, uniques = pd.factorize(train_fe[col])
    train_fe[col] = codes.astype(int)

    mapping = {val: i for i, val in enumerate(uniques)}
    test_fe[col] = test_fe[col].map(mapping).fillna(-1).astype(int)
    orig_fe[col] = orig_fe[col].map(mapping).fillna(-1).astype(int)

print("BIN_CAT_INT_COLS:", len(BIN_CAT_INT_COLS))

BIN_CAT_INT_COLS: 12


In [9]:
# ------------------------------------------------------------
# numeric aggregation by top categorical
# ------------------------------------------------------------

NUM_CAT_AGG_COLS = []

for cat_col in top_cat_cols:
    for num_col in top_num_cols:
        group_means = train_fe.groupby(cat_col)[num_col].mean()

        avg_name = f"{num_col}_avg_by_{cat_col}"
        diff_name = f"{num_col}_diff_{cat_col}"
        ratio_name = f"{num_col}_ratio_{cat_col}"

        for df in [train_fe, test_fe, orig_fe]:
            df[avg_name] = df[cat_col].map(group_means).astype(float)
            df[diff_name] = (df[num_col] - df[avg_name]).astype(float)
            df[ratio_name] = (df[num_col] / (df[avg_name] + 1e-6)).astype(float)

        NUM_CAT_AGG_COLS.extend([avg_name, diff_name, ratio_name])

NUM_CAT_AGG_COLS = list(dict.fromkeys(NUM_CAT_AGG_COLS))
print("NUM_CAT_AGG_COLS:", len(NUM_CAT_AGG_COLS))

NUM_CAT_AGG_COLS: 36


In [10]:
# ------------------------------------------------------------
# rounds, digits, decimals, bins
# ------------------------------------------------------------

round_config = {
    "Soil_Moisture": [0, -1],
    "Temperature_C": [-1],
    "Rainfall_mm": [0, -1, -2, -3],
    "Wind_Speed_kmh": [0, -1],
}

digit_config = {
    "Soil_Moisture": [-1, 0, 1, 2],
    "Temperature_C": [-1, 0, 1, 2],
    "Rainfall_mm": [-3, -2, -1, 0, 1, 2],
    "Wind_Speed_kmh": [-1, 0, 1, 2],
}

ROUND = []
for col, r_values in round_config.items():
    for r in r_values:
        feat = f"{col}_r{r}"
        for df in [train_fe, test_fe, orig_fe]:
            df[feat] = df[col].round(r)
        ROUND.append(feat)

DIGITS = []
for col, k_values in digit_config.items():
    for k in k_values:
        feat = f"{col}_d{k}"
        for df in [train_fe, test_fe, orig_fe]:
            df[feat] = ((df[col] * 10 ** k) % 10).astype(int)
        DIGITS.append(feat)

DECIMALS = []
for col in top_num_cols:
    feat = f"{col}_decimal"
    for df in [train_fe, test_fe, orig_fe]:
        df[feat] = (df[col] % 1).round(2)
    DECIMALS.append(feat)

BINS = []
for col in top_num_cols:
    feat = f"{col}_bin"
    train_fe[feat], bin_edges = pd.qcut(
        train_fe[col],
        q=10,
        labels=False,
        duplicates="drop",
        retbins=True,
    )

    for df in [test_fe, orig_fe]:
        df[col] = df[col].clip(bin_edges[0], bin_edges[-1])
        df[feat] = pd.cut(df[col], bins=bin_edges, labels=False, include_lowest=True).astype(int)

    BINS.append(feat)

print("ROUND:", len(ROUND), "DIGITS:", len(DIGITS), "DECIMALS:", len(DECIMALS), "BINS:", len(BINS))

ROUND: 9 DIGITS: 18 DECIMALS: 4 BINS: 4


In [11]:
# ------------------------------------------------------------
# pairs
# ------------------------------------------------------------

PAIRS = []

train_len = train_fe.shape[0]
test_len = test_fe.shape[0]
orig_len = orig_fe.shape[0]
combined_len = train_len + test_len + orig_len

for col1, col2 in combinations(top_cols, 2):
    name = f"{col1}__{col2}"

    combined_str = pd.concat(
        [
            train_fe[col1].astype(str) + "_" + train_fe[col2].astype(str),
            test_fe[col1].astype(str) + "_" + test_fe[col2].astype(str),
            orig_fe[col1].astype(str) + "_" + orig_fe[col2].astype(str),
        ],
        ignore_index=True,
    )

    combined_codes, _ = pd.factorize(combined_str)
    del combined_str
    gc.collect()

    unique_count = len(np.unique(combined_codes))
    if unique_count > (combined_len // 2) or unique_count <= 1:
        del combined_codes
        continue

    train_fe[name] = combined_codes[:train_len]
    test_fe[name] = combined_codes[train_len:train_len + test_len]
    orig_fe[name] = combined_codes[train_len + test_len:]

    PAIRS.append(name)
    del combined_codes
    gc.collect()

print("PAIRS:", len(PAIRS))

PAIRS: 9


In [12]:
# ------------------------------------------------------------
# Categorical dtype
# ------------------------------------------------------------

for df in [train_fe, test_fe, orig_fe]:
    for col in cat_cols:
        df[col] = df[col].astype("category")

In [13]:
# ============================================================
# Build train/test/orig matrices
# ============================================================

X_train = train_fe.drop([target, CFG.ID_COL], axis=1)
y_train = train_fe[target].astype(int)

X_orig = orig_fe.drop(target, axis=1)
y_orig = orig_fe[target].astype(int)

X_test = test_fe.drop(CFG.ID_COL, axis=1)

print("X_train:", X_train.shape)
print("X_orig :", X_orig.shape)
print("X_test :", X_test.shape)

DROP_COLS = PAIRS + NUM_CAT_AGG_COLS + BIN_CAT_INT_COLS
DROP_COLS = [c for c in DROP_COLS if c in X_train.columns]

te_features = X_train.columns.tolist()

print("DROP_COLS:", len(DROP_COLS))
print("te_features:", len(te_features))

X_train: (630000, 134)
X_orig : (10000, 134)
X_test : (270000, 134)
DROP_COLS: 57
te_features: 134


In [14]:
# ============================================================
# CV Training
# ============================================================

skf = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED)

oof_raw = np.zeros((len(X_train), 3), dtype=np.float32)
pred_raw = np.zeros((len(X_test), 3), dtype=np.float32)

fold_scores = []
feature_counts = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train), 1):
    print("\n" + "=" * 70)
    print(f"Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 70)

    X_tr_real = X_train.iloc[tr_idx].copy()
    y_tr_real = y_train.iloc[tr_idx].copy()

    X_val = X_train.iloc[va_idx].copy()
    y_val = y_train.iloc[va_idx].copy()

    X_tr_combined = pd.concat([X_tr_real, X_orig], axis=0).reset_index(drop=True)
    y_tr_combined = np.concatenate([y_tr_real.values, y_orig.values])

    encoder = TargetEncoder(
        target_type="multiclass",
        smooth="auto",
        cv=5,
        random_state=CFG.SEED,
    )

    X_tr_te = encoder.fit_transform(X_tr_combined[te_features], y_tr_combined)
    X_val_te = encoder.transform(X_val[te_features])
    X_test_te = encoder.transform(X_test[te_features])

    te_cols = []
    for col in te_features:
        for class_id in range(3):
            te_cols.append(f"TE_{col}_class{class_id}")

    X_tr_combined_te = X_tr_combined.copy()
    X_val_te_base = X_val.copy()
    X_test_te_base = X_test.copy()

    X_tr_combined_te[te_cols] = X_tr_te
    X_val_te_base[te_cols] = X_val_te
    X_test_te_base[te_cols] = X_test_te

    X_tr_final = X_tr_combined_te.drop(columns=DROP_COLS)
    X_val_final = X_val_te_base.drop(columns=DROP_COLS)
    X_te_final = X_test_te_base.drop(columns=DROP_COLS)

    sample_weights = compute_sample_weight("balanced", y_tr_combined)
    sample_weights[len(tr_idx):] *= CFG.ORIG_ROW_WEIGHT

    model = XGBClassifier(**CFG.XGB_PARAMS)
    model.fit(
        X_tr_final,
        y_tr_combined,
        eval_set=[(X_val_final, y_val)],
        sample_weight=sample_weights,
        verbose=200,
    )

    val_pred = model.predict_proba(X_val_final).astype(np.float32)
    test_pred = model.predict_proba(X_te_final).astype(np.float32)

    oof_raw[va_idx] = val_pred
    pred_raw += test_pred / CFG.N_SPLITS

    fold_ba = balanced_accuracy_score(y_val, np.argmax(val_pred, axis=1))
    fold_scores.append(float(fold_ba))
    feature_counts.append(int(X_tr_final.shape[1]))

    print(f"fold BA: {fold_ba:.9f}")
    print("n_features_final:", X_tr_final.shape[1])

    del (
        X_tr_real, y_tr_real, X_val, y_val,
        X_tr_combined, y_tr_combined,
        X_tr_te, X_val_te, X_test_te,
        X_tr_combined_te, X_val_te_base, X_test_te_base,
        X_tr_final, X_val_final, X_te_final,
        model, val_pred, test_pred
    )
    gc.collect()


raw_cv = balanced_accuracy_score(y_train, np.argmax(oof_raw, axis=1))
print("\nraw_cv:", raw_cv)
print("fold_scores:", fold_scores)


Fold 1/5
[0]	validation_0-mlogloss:1.02976
[200]	validation_0-mlogloss:0.06291
[400]	validation_0-mlogloss:0.05692
[600]	validation_0-mlogloss:0.05452
[800]	validation_0-mlogloss:0.05306
[1000]	validation_0-mlogloss:0.05200
[1200]	validation_0-mlogloss:0.05119
[1400]	validation_0-mlogloss:0.05057
[1600]	validation_0-mlogloss:0.04999
[1800]	validation_0-mlogloss:0.04957
[2000]	validation_0-mlogloss:0.04921
[2200]	validation_0-mlogloss:0.04890
[2400]	validation_0-mlogloss:0.04864
[2599]	validation_0-mlogloss:0.04842
fold BA: 0.977157413
n_features_final: 479

Fold 2/5
[0]	validation_0-mlogloss:1.02981
[200]	validation_0-mlogloss:0.06416
[400]	validation_0-mlogloss:0.05769
[600]	validation_0-mlogloss:0.05508
[800]	validation_0-mlogloss:0.05350
[1000]	validation_0-mlogloss:0.05239
[1200]	validation_0-mlogloss:0.05153
[1400]	validation_0-mlogloss:0.05090
[1600]	validation_0-mlogloss:0.05036
[1800]	validation_0-mlogloss:0.04994
[2000]	validation_0-mlogloss:0.04961
[2200]	validation_0-mloglo

In [15]:
# ============================================================
# Bias Tuning
# ============================================================

best_bias, tuned_cv, history = tune_bias_coordinate_descent(oof_raw, y_train.values)

oof_biased = apply_bias_to_proba(oof_raw, best_bias)
pred_biased = apply_bias_to_proba(pred_raw, best_bias)

biased_check = balanced_accuracy_score(y_train, np.argmax(oof_biased, axis=1))

print("best_bias:", best_bias)
print("tuned_cv:", tuned_cv)
print("biased_check:", biased_check)
print("gain:", tuned_cv - raw_cv)

print("\nClassification report after bias:")
print(
    classification_report(
        y_train,
        np.argmax(oof_biased, axis=1),
        target_names=["Low", "Medium", "High"],
    )
)

bias step=1.0000 score=0.979572987 bias=[ 0. -1.  0.]
bias step=1.0000 score=0.979870190 bias=[-1. -1.  0.]
bias step=0.2000 score=0.979887888 bias=[-1.2 -1.   0. ]
bias step=0.2000 score=0.980124448 bias=[-1.2 -1.2  0. ]
bias step=0.2000 score=0.980142146 bias=[-1.4 -1.2  0. ]
bias step=0.1000 score=0.980142512 bias=[-1.3 -1.2  0. ]
bias step=0.0500 score=0.980147736 bias=[-1.35 -1.2   0.  ]
bias step=0.0200 score=0.980150305 bias=[-1.37 -1.2   0.  ]
bias step=0.0100 score=0.980154147 bias=[-1.36 -1.2   0.  ]
bias step=0.0050 score=0.980158379 bias=[-1.36  -1.195  0.   ]
bias step=0.0050 score=0.980159195 bias=[-1.355 -1.195  0.   ]
bias step=0.0020 score=0.980159688 bias=[-1.357 -1.195  0.   ]
bias step=0.0020 score=0.980161576 bias=[-1.357 -1.193  0.   ]
best_bias: [-1.357 -1.193  0.   ]
tuned_cv: 0.9801615755373589
biased_check: 0.9801615755373589
gain: 0.0018953658332786505

Classification report after bias:
              precision    recall  f1-score   support

         Low      

In [16]:
# ============================================================
# Save artifacts
# ============================================================

np.save(CFG.SAVE_DIR / "oof_ashish_xgb_orig_te_raw.npy", oof_raw)
np.save(CFG.SAVE_DIR / "pred_ashish_xgb_orig_te_raw.npy", pred_raw)

np.save(CFG.SAVE_DIR / "oof_ashish_xgb_orig_te_bias.npy", oof_biased)
np.save(CFG.SAVE_DIR / "pred_ashish_xgb_orig_te_bias.npy", pred_biased)

np.save(CFG.SAVE_DIR / "best_class_bias.npy", best_bias.astype(np.float32))

submission = pd.DataFrame(
    {
        CFG.ID_COL: test[CFG.ID_COL],
        CFG.TARGET: pd.Series(np.argmax(pred_biased, axis=1)).map(CFG.INV_TARGET_MAP),
    }
)
submission.to_csv(CFG.SAVE_DIR / "submission_ashish_xgb_orig_te_bias.csv", index=False)

print("submission distribution:")
print(submission[CFG.TARGET].value_counts(normalize=True))

cv_result = {
    "exp_id": CFG.EXP_ID,
    "exp_name": CFG.EXP_NAME,
    "raw_cv": float(raw_cv),
    "tuned_cv": float(tuned_cv),
    "biased_check": float(biased_check),
    "gain_bias": float(tuned_cv - raw_cv),
    "best_bias": [float(x) for x in best_bias],
    "fold_scores": fold_scores,
    "feature_counts": feature_counts,
    "orig_row_weight": float(CFG.ORIG_ROW_WEIGHT),
    "xgb_params": CFG.XGB_PARAMS,
    "features": {
        "cat_cols": cat_cols,
        "num_cols": num_cols,
        "top_cat_cols": top_cat_cols,
        "top_num_cols": top_num_cols,
        "NGRAM_COLS": NGRAM_COLS,
        "BIN_CAT_INT_COLS_count": len(BIN_CAT_INT_COLS),
        "NUM_CAT_AGG_COLS_count": len(NUM_CAT_AGG_COLS),
        "ROUND_count": len(ROUND),
        "DIGITS_count": len(DIGITS),
        "DECIMALS_count": len(DECIMALS),
        "BINS_count": len(BINS),
        "PAIRS_count": len(PAIRS),
        "DROP_COLS_count": len(DROP_COLS),
        "te_features_count": len(te_features),
    },
    "source": {
        "url": "https://www.kaggle.com/code/rawashishsin/s6e4-highest-score-xgboost-cv-0-98109",
        "note": "shared notebook reports raw OOF 0.978266 and tuned OOF 0.980162; title/LB indicates 0.98109",
    },
}

save_json(cv_result, CFG.SAVE_DIR / "cv_result.json")

submission distribution:
Irrigation_Need
Low       0.590293
Medium    0.371504
High      0.038204
Name: proportion, dtype: float64


In [17]:
# ============================================================
# Correlation vs existing OOF
# ============================================================

corr_rows = []
for name, path in CFG.REF_OOF.items():
    if os.path.exists(path):
        ref = np.load(path)
        corr_rows.append(
            {
                "ref": name,
                "raw_corr_mean": mean_raw_corr(oof_biased, ref),
                "rank_corr_mean": mean_rank_corr(oof_biased, ref),
            }
        )
    else:
        print("missing ref:", name, path)

corr_df = pd.DataFrame(corr_rows)
if len(corr_df):
    corr_df = corr_df.sort_values("rank_corr_mean", ascending=False)
    corr_df.to_csv(CFG.SAVE_DIR / "corr_vs_existing_oof.csv", index=False)
    display(corr_df)

print("saved to:", CFG.SAVE_DIR)
print("Done.")

,ref,raw_corr_mean,rank_corr_mean
5,057,0.991391,0.964873
1,046b,0.991471,0.960947
3,053,0.992540,0.956727
2,025,0.991512,0.949017
0,018,0.989559,0.946777
4,056,0.992101,0.943018
6,060,0.972215,0.927905


saved to: /kaggle/working/exp_20260428_061_ashish_xgb_orig_te_bias
Done.
